# eBay Listings Scraper With Playwright

This notebook uses a real browser through **Playwright** instead of raw `requests`, which is much more reliable for eBay search pages.

It is designed to:

- collect large numbers of eBay listing URLs
- scrape listing pages for structured fields
- build the features you asked for
- save progress incrementally so long runs are safer

## Target features

- `category`
- `subcategory`
- `brand`
- `condition`
- `age_months`
- `description_length`
- `image_count`
- `day_of_week`
- `day_of_month`
- sentence-transformer embeddings

## Extra features

- `title`
- `title_length`
- `price`
- `currency`
- `seller_name`
- `shipping_text`
- `item_number`
- `location`
- `listing_url`
- `listing_date_text`
- `week_of_month`
- `hour_of_day`


## Install dependencies

Run these cells once in the notebook environment.


In [ ]:
%pip install -q pandas numpy beautifulsoup4 lxml tqdm sentence-transformers playwright scikit-learn


In [ ]:
!playwright install chromium


In [ ]:
import json
import math
import os
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


## Configuration

Increase `SEARCH_PAGES_PER_SEED` and add more seed queries if you want to push toward 10k rows.


In [ ]:
SEARCH_SEEDS = [
    # Smartphones — strong brand + fast condition depreciation, $50-$1500
    "iphone",
    "samsung galaxy phone",
    "android smartphone",

    # Computing — strong brand, age matters a lot, $100-$2000
    "macbook",
    "laptop",
    "ipad",

    # Audio & accessories — brand range from budget to premium, $20-$400
    "wireless headphones",
    "bluetooth speaker",
    "apple watch",

    # Gaming — condition + completeness critical, $20-$600
    "playstation 5",
    "nintendo switch",
    "video game lot",
    "xbox series",

    # Fashion — brand premium is strong, condition less sharp than electronics, $10-$800
    "nike shoes",
    "levi jeans",
    "north face jacket",
    "designer handbag",
    "vintage denim jacket",

    # Home & kitchen — brand matters, condition moderate, $20-$500
    "dyson vacuum",
    "air fryer",
    "kitchenaid mixer",
    "coffee machine",

    # Sports & outdoors — condition + brand, $15-$600
    "golf clubs",
    "road bike",
    "gym weights",

    # Collectibles — condition + rarity dominate, huge price spread $5-$2000+
    "pokemon cards",
    "lego set",
    "vinyl record",
    "vintage camera",
    "funko pop",

    # Books & media — low price, condition matters but less dramatically, $5-$100
    "textbook",
    "hardcover book",
]

# 32 seeds x 10 pages = ~320 search pages -> ~13k-16k URLs before dedup.
# Raise DETAIL_PAGE_LIMIT to scrape more detail pages than the old 3k default.
SEARCH_PAGES_PER_SEED = 10
DETAIL_PAGE_LIMIT = 10000
HEADLESS = False
SLOW_MO_MS = 0
NAV_TIMEOUT_MS = 45000
MIN_DELAY = 1.2
MAX_DELAY = 2.5
SAVE_EVERY_N_LISTINGS = 50
SOLD_ONLY = True
BUILD_EMBEDDINGS = True
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_PCA_COMPONENTS = 16
COMPARABLE_GROUP_COLUMNS = ["category", "subcategory", "brand"]
UNKNOWN_BRAND_BUCKET = "__UNKNOWN_BRAND__"
CONDITION_ORDER = {
    "new": 5,
    "like new": 4,
    "open box": 4,
    "certified refurbished": 4,
    "excellent - refurbished": 4,
    "very good - refurbished": 3,
    "good": 3,
    "good - refurbished": 3,
    "fair": 2,
    "acceptable": 2,
    "for parts or not working": 1,
    "poor": 1,
}


In [ ]:
def jitter_sleep():
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))


def clean_text(value):
    if value is None:
        return None
    value = re.sub(r"\s+", " ", str(value)).strip()
    return value or None


def safe_filename(value):
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", value).strip("_").lower()


def save_debug_artifacts(name, html=None, screenshot_bytes=None):
    stem = safe_filename(name)
    if html is not None:
        (OUTPUT_DIR / f"{stem}.html").write_text(html, encoding="utf-8")
    if screenshot_bytes is not None:
        (OUTPUT_DIR / f"{stem}.png").write_bytes(screenshot_bytes)


def month_diff(later, earlier):
    if later is None or earlier is None:
        return None
    return round((later - earlier).days / 30.4375, 3)


def clip_upper_quantile(series, q=0.99):
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().sum() == 0:
        return numeric
    upper = numeric.quantile(q)
    return numeric.clip(upper=upper)


def week_of_month(ts):
    if pd.isna(ts):
        return None
    return int(math.ceil(ts.day / 7))


def parse_date_text(value):
    if not value:
        return None

    value = clean_text(value)
    if not value:
        return None

    patterns = [
        r"Listed\s+(.*)",
        r"Posted\s+(.*)",
        r"Ends\s+(.*)",
        r"Starting bid\s+(.*)",
    ]

    for pattern in patterns:
        match = re.search(pattern, value, flags=re.IGNORECASE)
        if match:
            value = match.group(1).strip()
            break

    value = value.replace("at ", "")
    value = value.replace("GMT", "+0000")
    value = value.replace("BST", "+0100")

    for fmt in [
        "%b %d, %Y %H:%M:%S %z",
        "%d %b, %Y %H:%M:%S %z",
        "%b %d, %Y %H:%M %z",
        "%d %b, %Y %H:%M %z",
        "%b %d, %Y",
        "%d %b %Y",
    ]:
        try:
            dt = datetime.strptime(value, fmt)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)
            return dt.astimezone(timezone.utc)
        except ValueError:
            pass

    return None


def normalize_condition(value):
    value = clean_text(value)
    if not value:
        return None
    lower = value.lower()
    for key in CONDITION_ORDER:
        if key in lower:
            return key
    return lower


def to_cyclical(series, period):
    numeric = pd.to_numeric(series, errors="coerce")
    angle = 2 * np.pi * (numeric / period)
    return np.sin(angle), np.cos(angle)


def make_comparable_key(df, columns):
    parts = []
    for col in columns:
        parts.append(df[col].fillna("unknown").astype(str).str.lower().str.strip())
    return parts[0].str.cat(parts[1:], sep="|") if len(parts) > 1 else parts[0]


def extract_age_months_from_specifics(specifics_json, reference_dt):
    # Estimate item age in months from eBay item specifics year fields.
    # Falls back to None when no year field is present (XGBoost handles nulls fine).
    if not specifics_json:
        return None
    try:
        specifics = json.loads(specifics_json) if isinstance(specifics_json, str) else specifics_json
    except Exception:
        return None
    for key in ("year", "model year", "manufacture year", "release year"):
        val = specifics.get(key)
        if val:
            m = re.search(r"\b(19|20)\d{2}\b", str(val))
            if m:
                year = int(m.group(0))
                ref = reference_dt if reference_dt else datetime.now(timezone.utc)
                if hasattr(ref, "to_pydatetime"):
                    ref = ref.to_pydatetime()
                age = round((ref.year - year) * 12 + ref.month, 1)
                return age if age >= 0 else None
    return None


def add_prior_comparable_stats(df, group_col="comparable_key", price_col="sale_price", time_col="listing_dt"):
    work = df.copy().sort_values([time_col, "scraped_at"], na_position="last").reset_index(drop=False)

    median_vals = pd.Series(np.nan, index=work.index)
    mean_vals = pd.Series(np.nan, index=work.index)
    std_vals = pd.Series(np.nan, index=work.index)
    count_vals = pd.Series(0.0, index=work.index)

    for _, grp in work.groupby(group_col, sort=False):
        prices = grp[price_col].reset_index(drop=True)
        exp = prices.expanding()
        median_vals.loc[grp.index] = exp.median().shift(1).values
        mean_vals.loc[grp.index] = exp.mean().shift(1).values
        std_vals.loc[grp.index] = exp.std().shift(1).values
        count_vals.loc[grp.index] = exp.count().shift(1).fillna(0).values

    work["comparable_median_price"] = median_vals
    work["comparable_mean_price"] = mean_vals
    work["comparable_stdev_price"] = std_vals
    work["comparable_count"] = count_vals
    # Exact sold_ratio needs both sold and unsold inventory streams.
    # In sold-only mode we leave it missing instead of leaking or inventing a denominator.
    work["comparable_sold_ratio"] = np.nan
    work["avg_offer_ratio_in_category"] = np.nan
    return work.sort_values("index").drop(columns=["index"])


def fit_embedding_pca(texts, n_components=16, model_name=EMBEDDING_MODEL_NAME):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    embeddings = np.asarray(embeddings)
    if embeddings.ndim == 1:
        embeddings = embeddings.reshape(1, -1)
    pca = PCA(n_components=min(n_components, embeddings.shape[0], embeddings.shape[1]))
    reduced = pca.fit_transform(embeddings)
    cols = [f"description_embedding_pca_{i}" for i in range(reduced.shape[1])]
    return pd.DataFrame(reduced, columns=cols), model, pca


def out_of_fold_target_encode(
    df,
    column,
    target="sale_price",
    n_splits=5,
    unknown_bucket=None,
):
    work = df[[column, target, "listing_dt"]].copy()
    if unknown_bucket is not None:
        work[column] = work[column].fillna(unknown_bucket)
    else:
        work[column] = work[column].fillna("__MISSING__")

    work = work.sort_values("listing_dt", na_position="last").reset_index()
    folds = np.array_split(work.index.to_numpy(), min(n_splits, max(len(work), 1)))
    encoded = pd.Series(index=work.index, dtype="float64")
    global_mean = work[target].mean()

    for fold in folds:
        train_mask = ~work.index.isin(fold)
        train = work.loc[train_mask]
        valid = work.loc[fold]
        mapping = train.groupby(column)[target].mean()
        encoded.loc[fold] = valid[column].map(mapping).fillna(global_mean).values

    work[f"{column}_target_encoded"] = encoded
    work = work.sort_values("index")
    return work[f"{column}_target_encoded"].reset_index(drop=True)


## Browser startup


In [ ]:
playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=HEADLESS, slow_mo=SLOW_MO_MS)
context = await browser.new_context(
    viewport={"width": 1440, "height": 2200},
    user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    locale="en-US",
    timezone_id="Europe/London",
)
page = await context.new_page()
page.set_default_timeout(NAV_TIMEOUT_MS)
print("Browser ready")


## Search result scraping


In [ ]:
def build_search_url(query, page_num):
    query_encoded = query.replace(" ", "+")
    url = f"https://www.ebay.com/sch/i.html?_nkw={query_encoded}&_pgn={page_num}&_sop=10"
    if SOLD_ONLY:
        url += "&LH_Sold=1&LH_Complete=1"
    return url


async def maybe_dismiss_overlays(page):
    for selector in [
        'button[aria-label="Close overlay"]',
        'button[aria-label="Close dialog"]',
        'button[aria-label="Accept"]',
        '#gdpr-banner-accept',
    ]:
        try:
            if await page.locator(selector).count() > 0:
                await page.locator(selector).first.click(timeout=1500)
                jitter_sleep()
        except Exception:
            pass


async def extract_search_cards(page):
    await page.wait_for_load_state("domcontentloaded")
    await maybe_dismiss_overlays(page)
    try:
        await page.wait_for_selector("#srp-river-results, .srp-river-results, a[href*='/itm/']", timeout=10000)
    except PlaywrightTimeoutError:
        return [], {"reason": "selector_timeout"}

    cards = await page.evaluate(
        """
        () => {
          const anchors = Array.from(document.querySelectorAll("a[href*='/itm/']"));
          const seen = new Set();
          const rows = [];

          const normalizeUrl = (href) => {
            try {
              const u = new URL(href, location.origin);
              if (!u.pathname.includes('/itm/')) return null;
              return `${u.origin}${u.pathname}`;
            } catch {
              return null;
            }
          };

          const getText = (node, selectors) => {
            for (const selector of selectors) {
              const el = node.querySelector(selector);
              const text = el?.innerText?.trim();
              if (text) return text;
            }
            return null;
          };

          for (const a of anchors) {
            const rawUrl = a.getAttribute('href') || a.href;
            const listingUrl = normalizeUrl(rawUrl);
            if (!listingUrl || seen.has(listingUrl)) continue;

            const container =
              a.closest('li') ||
              a.closest('div[role="listitem"]') ||
              a.closest('div[class*="result"]') ||
              a.closest('div[class*="item"]') ||
              a.parentElement;

            const title =
              a.innerText?.trim() ||
              getText(container || document, [
                'h3',
                'h2',
                '[role="heading"]',
                'span[class*="title"]',
                'div[class*="title"]'
              ]);

            if (!title || title.toLowerCase() === 'shop on ebay') continue;

            const priceText = getText(container || document, [
              '.s-item__price',
              '[class*="price"]',
              'span[aria-label*="$"]'
            ]);

            const shippingText = getText(container || document, [
              '.s-item__shipping',
              '.s-item__logisticsCost',
              '[class*="shipping"]'
            ]);

            const subtitle = getText(container || document, [
              '.s-item__subtitle',
              '[class*="subtitle"]'
            ]);

            const condition = getText(container || document, [
              '.SECONDARY_INFO',
              '[class*="condition"]'
            ]);

            const imageEl = (container || document).querySelector('img');
            const imageUrl = imageEl?.src || imageEl?.getAttribute('data-src') || null;

            seen.add(listingUrl);
            rows.push({
              title,
              listing_url: listingUrl,
              price_text: priceText,
              shipping_text: shippingText,
              subtitle,
              condition,
              image_url: imageUrl
            });
          }

          return rows;
        }
        """
    )
    debug = {
        "title": await page.title(),
        "url": page.url,
        "card_count": len(cards),
    }
    return cards, debug


SEARCH_RESULT_COLUMNS = [
    "title",
    "listing_url",
    "price_text",
    "shipping_text",
    "subtitle",
    "condition",
    "image_url",
    "seed_query",
    "search_page",
]


async def collect_search_results(page, seeds, pages_per_seed):
    rows = []
    for seed in tqdm(seeds, desc="Seed queries"):
        for page_num in range(1, pages_per_seed + 1):
            url = build_search_url(seed, page_num)
            try:
                await page.goto(url, wait_until="domcontentloaded")
                jitter_sleep()
                cards, debug = await extract_search_cards(page)
            except Exception as exc:
                print(f"Search error for seed={seed!r}, page={page_num}: {exc}")
                break

            if not cards:
                html = await page.content()
                screenshot_bytes = await page.screenshot(full_page=True)
                artifact_name = f"debug_{seed}_page_{page_num}"
                save_debug_artifacts(artifact_name, html=html, screenshot_bytes=screenshot_bytes)
                print(
                    f"No cards found for seed={seed!r}, page={page_num}. "
                    f"page_title={debug.get('title')!r} url={debug.get('url')!r}. "
                    f"Saved debug artifacts: output/{safe_filename(artifact_name)}.html/.png"
                )
                break

            for card in cards:
                card["seed_query"] = seed
                card["search_page"] = page_num
                card["scraped_at"] = datetime.now(timezone.utc).isoformat()
            rows.extend(cards)

            if len(rows) % 200 == 0:
                pd.DataFrame(rows).drop_duplicates(subset=["listing_url"]).to_csv(
                    OUTPUT_DIR / "ebay_search_results_partial.csv", index=False
                )
    if not rows:
        return pd.DataFrame(columns=SEARCH_RESULT_COLUMNS)

    df = pd.DataFrame(rows)
    for col in SEARCH_RESULT_COLUMNS:
        if col not in df.columns:
            df[col] = pd.Series([None] * len(df), dtype="object")
    return df.drop_duplicates(subset=["listing_url"]).reset_index(drop=True)


In [ ]:
search_df = await collect_search_results(page, SEARCH_SEEDS, SEARCH_PAGES_PER_SEED)
search_df.to_csv(OUTPUT_DIR / "ebay_search_results.csv", index=False)
print("Search rows:", len(search_df))
print("Search columns:", list(search_df.columns))
if "listing_url" in search_df.columns:
    print("Non-null listing URLs:", int(search_df["listing_url"].notna().sum()))
else:
    print("No listing_url column was created.")
search_df.head()


## Listing detail scraping

This step visits each listing page and extracts richer fields.


In [ ]:
def parse_price_to_float(price_text):
    if not price_text:
        return None
    match = re.search(r"([\d,.]+)", price_text.replace(",", ""))
    return float(match.group(1)) if match else None


def infer_currency(price_text):
    if not price_text:
        return None
    if "$" in price_text:
        return "USD"
    if "GBP" in price_text or "£" in price_text:
        return "GBP"
    if "EUR" in price_text or "€" in price_text:
        return "EUR"
    return None


async def scrape_listing_detail(page, url):
    await page.goto(url, wait_until="domcontentloaded")
    jitter_sleep()
    await maybe_dismiss_overlays(page)
    html = await page.content()
    soup = BeautifulSoup(html, "lxml")

    title = clean_text(
        soup.select_one("h1.x-item-title__mainTitle span") and
        soup.select_one("h1.x-item-title__mainTitle span").get_text(" ", strip=True)
    )

    breadcrumbs = [
        clean_text(a.get_text(" ", strip=True))
        for a in soup.select('nav[aria-label="Breadcrumb"] a, [aria-label="Breadcrumb"] a')
    ]
    breadcrumbs = [x for x in breadcrumbs if x]

    seller_name = None
    for selector in [
        '.x-sellercard-atf__info__about-seller a',
        '.ux-seller-section__item--seller a',
        '[data-testid="ux-seller-section__item--seller"] a',
    ]:
        el = soup.select_one(selector)
        if el:
            seller_name = clean_text(el.get_text(" ", strip=True))
            break

    image_urls = set()
    for img in soup.select("img"):
        src = img.get("src") or img.get("data-src") or img.get("data-zoom-src")
        if src and "ebayimg.com" in src:
            image_urls.add(src)

    item_specifics = {}
    for row in soup.select(".ux-layout-section-evo__row, .ux-layout-section__row"):
        text = clean_text(row.get_text(" ", strip=True))
        if text and ":" in text:
            key, value = text.split(":", 1)
            key = clean_text(key).lower()
            value = clean_text(value)
            if key and value and len(key) < 80:
                item_specifics[key] = value

    json_ld = []
    for script in soup.select('script[type="application/ld+json"]'):
        raw = script.string or script.get_text(strip=True)
        if not raw:
            continue
        try:
            json_ld.append(json.loads(raw))
        except Exception:
            pass

    product_block = {}
    for block in json_ld:
        if isinstance(block, dict) and block.get("@type") == "Product":
            product_block = block
            break

    desc = clean_text(product_block.get("description"))
    if not desc:
        meta_desc = soup.select_one('meta[name="description"]')
        desc = clean_text(meta_desc.get("content")) if meta_desc else None

    listing_date_candidates = []
    for selector in [
        ".ux-labels-values--listed-on .ux-textspans",
        ".ux-labels-values--ends .ux-textspans",
        ".vi-bboxrev-dsplb",
        ".ux-textspans--BOLD",
    ]:
        for el in soup.select(selector):
            txt = clean_text(el.get_text(" ", strip=True))
            if txt:
                listing_date_candidates.append(txt)

    listing_date_text = next((x for x in listing_date_candidates if re.search(r"(Listed|Ends|[A-Z][a-z]{2}\s+\d{1,2})", x)), None)
    item_number = item_specifics.get("ebay item number") or item_specifics.get("item number")
    location = item_specifics.get("located in") or item_specifics.get("location")
    brand = item_specifics.get("brand") or clean_text(product_block.get("brand"))
    condition = item_specifics.get("condition")

    result = {
        "listing_url": url,
        "title": title,
        "category": breadcrumbs[0] if breadcrumbs else None,
        "subcategory": breadcrumbs[-1] if len(breadcrumbs) > 1 else None,
        "brand": brand,
        "condition": condition,
        "description": desc,
        "description_length": len(desc) if desc else 0,
        "image_count": len(image_urls),
        "seller_name": seller_name,
        "listing_date_text": listing_date_text,
        "item_number": item_number,
        "location": location,
        "detail_scraped_at": datetime.now(timezone.utc).isoformat(),
        "raw_item_specifics_json": json.dumps(item_specifics),
    }
    return result


In [ ]:
detail_rows = []
if "listing_url" not in search_df.columns:
    to_scrape = []
else:
    to_scrape = search_df["listing_url"].dropna().tolist()[:DETAIL_PAGE_LIMIT]

print("Listing URLs queued for detail scraping:", len(to_scrape))

for idx, url in enumerate(tqdm(to_scrape, desc="Listing details"), start=1):
    try:
        row = await scrape_listing_detail(page, url)
        detail_rows.append(row)
    except Exception as exc:
        detail_rows.append({"listing_url": url, "detail_error": str(exc)})

    if idx % SAVE_EVERY_N_LISTINGS == 0:
        pd.DataFrame(detail_rows).to_csv(OUTPUT_DIR / "ebay_listing_details_partial.csv", index=False)

detail_df = pd.DataFrame(detail_rows)
if "listing_url" not in detail_df.columns:
    detail_df["listing_url"] = pd.Series(dtype="object")
detail_df = detail_df.drop_duplicates(subset=["listing_url"])
detail_df.to_csv(OUTPUT_DIR / "ebay_listing_details.csv", index=False)
print("Detail rows:", len(detail_df))
detail_df.head()


## Merge and feature engineering


In [ ]:
df = search_df.merge(detail_df, on="listing_url", how="left", suffixes=("_search", ""))

if "title" not in df.columns and "title_search" in df.columns:
    df["title"] = df["title_search"]
elif "title" in df.columns and "title_search" in df.columns:
    df["title"] = df["title"].fillna(df["title_search"])

if "condition" not in df.columns and "condition_search" in df.columns:
    df["condition"] = df["condition_search"]
elif "condition" in df.columns and "condition_search" in df.columns:
    df["condition"] = df["condition"].fillna(df["condition_search"])

required_columns = [
    "category",
    "subcategory",
    "brand",
    "condition",
    "description",
    "description_length",
    "image_count",
    "listing_date_text",
    "title",
    "price_text",
    "shipping_text",
    "seller_name",
    "item_number",
    "location",
    "scraped_at",
    "detail_scraped_at",
]

for col in required_columns:
    if col not in df.columns:
        df[col] = pd.Series([None] * len(df), dtype="object")

df["scraped_at"] = pd.to_datetime(df["scraped_at"], utc=True, errors="coerce")
df["detail_scraped_at"] = pd.to_datetime(df["detail_scraped_at"], utc=True, errors="coerce")
df["listing_dt"] = df["listing_date_text"].apply(parse_date_text)
df["listing_dt"] = pd.to_datetime(df["listing_dt"], utc=True, errors="coerce")

now_utc = datetime.now(timezone.utc)
df["age_months"] = df.apply(
    lambda r: extract_age_months_from_specifics(r.get("raw_item_specifics_json"), now_utc),
    axis=1,
)

df["title"] = df["title"].apply(clean_text)
df["description"] = df["description"].apply(clean_text)
df["brand"] = df["brand"].apply(clean_text)
df["condition"] = df["condition"].apply(clean_text)
df["category"] = df["category"].apply(clean_text)
df["subcategory"] = df["subcategory"].apply(clean_text)
df["shipping_text"] = df["shipping_text"].apply(clean_text)
df["location"] = df["location"].apply(clean_text)

df["title_length"] = df["title"].fillna("").str.len()
df["description_length"] = df["description"].fillna("").str.len()
df["sale_price"] = df["price_text"].apply(parse_price_to_float)
df["currency"] = df["price_text"].apply(infer_currency)
df["title_description_text"] = df["title"].fillna("") + ". " + df["description"].fillna("")
df["brand"] = df["brand"].fillna(UNKNOWN_BRAND_BUCKET)

df["condition_normalized"] = df["condition"].apply(normalize_condition)
df["condition_ordinal"] = df["condition_normalized"].map(CONDITION_ORDER).fillna(0).astype(int)

df["age_months"] = clip_upper_quantile(df["age_months"], q=0.99)

df["day_of_week"] = df["listing_dt"].dt.dayofweek
df["month"] = df["listing_dt"].dt.month
df["day_of_week_sin"], df["day_of_week_cos"] = to_cyclical(df["day_of_week"], 7)
df["month_sin"], df["month_cos"] = to_cyclical(df["month"], 12)

df["comparable_key"] = make_comparable_key(df, COMPARABLE_GROUP_COLUMNS)
df = add_prior_comparable_stats(df, group_col="comparable_key", price_col="sale_price", time_col="listing_dt")

# Leakage-aware target encodings should be recomputed inside CV folds for real modeling.
# These columns are included as a dataset convenience baseline.
df["category_target_encoded"] = out_of_fold_target_encode(df, "category", target="sale_price")
df["subcategory_target_encoded"] = out_of_fold_target_encode(df, "subcategory", target="sale_price")
df["brand_target_encoded"] = out_of_fold_target_encode(
    df,
    "brand",
    target="sale_price",
    unknown_bucket=UNKNOWN_BRAND_BUCKET,
)

print("Merged rows:", len(df))
df.head()


## Optional sentence-transformer embeddings

Leave `BUILD_EMBEDDINGS = False` for the first full scrape. Turn it on after you confirm you have real rows.


In [ ]:
if BUILD_EMBEDDINGS:
    texts = df["title_description_text"].fillna("").tolist()
    if len(texts) == 0:
        print("Skipping embeddings because the dataset is empty.")
    else:
        embedding_df, embedding_model, embedding_pca = fit_embedding_pca(
            texts,
            n_components=EMBEDDING_PCA_COMPONENTS,
            model_name=EMBEDDING_MODEL_NAME,
        )
        embedding_df.index = df.index
        df = pd.concat([df, embedding_df], axis=1)
        print("Embedding PCA columns added:", embedding_df.shape[1])
else:
    print("Embeddings disabled for now.")


In [ ]:
preferred_columns = [
    "listing_url",
    "seed_query",
    "search_page",
    "category",
    "subcategory",
    "brand",
    "category_target_encoded",
    "subcategory_target_encoded",
    "brand_target_encoded",
    "condition",
    "condition_normalized",
    "condition_ordinal",
    "age_months",
    "description_length",
    "image_count",
    "sale_price",
    "comparable_median_price",
    "comparable_mean_price",
    "comparable_stdev_price",
    "comparable_count",
    "comparable_sold_ratio",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
    "avg_offer_ratio_in_category",
    "title_length",
    "currency",
    "seller_name",
    "shipping_text",
    "item_number",
    "location",
    "title",
    "description",
    "listing_date_text",
    "listing_dt",
    "price_text",
    "image_url",
    "subtitle",
    "scraped_at",
    "detail_scraped_at",
    "comparable_key",
    "raw_item_specifics_json",
    "detail_error",
]

final_columns = [c for c in preferred_columns if c in df.columns] + [c for c in df.columns if c not in preferred_columns]
final_df = df[final_columns].copy()

final_df.to_csv(OUTPUT_DIR / "ebay_playwright_features.csv", index=False)
final_df.to_parquet(OUTPUT_DIR / "ebay_playwright_features.parquet", index=False)

print("Saved files:")
print(OUTPUT_DIR / "ebay_search_results.csv")
print(OUTPUT_DIR / "ebay_listing_details.csv")
print(OUTPUT_DIR / "ebay_playwright_features.csv")
print(OUTPUT_DIR / "ebay_playwright_features.parquet")
print("Final rows:", len(final_df))
final_df.head()


## Cleanup

Run this when you are done.


In [ ]:
await page.close()
await context.close()
await browser.close()
await playwright.stop()
print("Browser closed")
